In [1]:
import os
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langsmith import traceable  # Import the LangSmith tracing decorator
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
import os
os.environ['OPENAI_API_BASE'] = "https://openai.vocareum.com/v1"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "LangSmith_Agent_Tracing_Demo_19Jul_v2"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

In [3]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b


In [4]:
@traceable(name="Agent Pipeline Execution", run_type="chain")
def run_agent_pipeline(query: str):
    # Initialize model and tools
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    tools = [add_numbers, multiply_numbers]
    SYSTEM_PROMPT = """ You are a helpful mathematics wizard who uses tools to do maths calculations. Only use the tools and no
    other information. Respond politely and if you do not know, say you do not know.
    """
    prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT),
                                               ("human",  "{query}"),
                                               MessagesPlaceholder(variable_name="agent_scratchpad")])
    

    agent = create_tool_calling_agent(llm=model, tools=tools, prompt=prompt)
    executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
    result = executor.invoke({"query": f"{query}"})
    
    # Return the final message content
    return result.get('output')

In [5]:
user_query = "What is 15 plus 35, and then multiplied by 3?"
print(f"Starting pipeline with query: '{user_query}'...")
final_output = run_agent_pipeline(user_query)

Starting pipeline with query: 'What is 15 plus 35, and then multiplied by 3?'...


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `add_numbers` with `{'a': 15, 'b': 35}`


50
Invoking: `multiply_numbers` with `{'a': 50, 'b': 3}`


15015 plus 35 is 50, and when multiplied by 3, the result is 150.

> Finished chain.
